# Exercise 2: where do the eight labels fail?

**On your own, at home.** Sixty tickets you have not seen yet: T061 to T120.
The cells marked `GIVEN` run as they are; in the others, write the code under
each comment. A full run costs about $0.07.

In [ ]:
# GIVEN: run this cell as it is.
# It loads the key from .env, creates the client, downloads the 120 tickets once,
# and defines the test set (TEST) and the pool your examples come from (POOL).
import csv
import datetime
import re
import urllib.request
from collections import Counter
from pathlib import Path

import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"
PRICE_IN, PRICE_OUT = 1.00, 5.00   # $ per million tokens (Anthropic, 1 Oct 2026)

URL = ("https://github.com/TangoAI-AlbertSchool-Data-Eng-Cloud/albert-marketplace"
       "/releases/download/tickets-v1.0.0/tickets_v1.csv")
path = Path("data/tickets_v1.csv")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, path)
tickets = {r["ticket_id"]: r for r in csv.DictReader(path.open(encoding="utf-8"))}

LABELS = ["billing", "delivery", "returns_refunds", "account", "seller", "product", "information", "other"]
TEST = ["T005", "T008", "T016", "T018", "T023", "T024", "T032", "T034", "T042", "T045"]
POOL = [f"T{i:03d}" for i in range(1, 61) if f"T{i:03d}" not in TEST]   # the other 50 of batch 1
print(len(tickets), "tickets | test set:", len(TEST), "| pool:", len(POOL), "| run on", datetime.date.today())

In [ ]:
# GIVEN: the plumbing. Read it once; you do not need to change it.
spent = Counter()                      # every token this notebook uses


def text_of(t):
    """A ticket as the model reads it: its subject, a blank line, its body."""
    return f"Subject: {t['subject']}\n\n{t['body']}"


def ask(user, system, max_tokens=1000, **extra):
    """One call to the model; counts the tokens it used."""
    r = client.messages.create(model=MODEL, max_tokens=max_tokens, system=system,
                               messages=[{"role": "user", "content": user}], **extra)
    spent["in"] += r.usage.input_tokens
    spent["out"] += r.usage.output_tokens
    return r


def answer_text(r):
    """The visible answer: the text blocks joined, thinking blocks skipped."""
    return "".join(b.text for b in r.content if b.type == "text")


def parse(text, labels=LABELS):
    """The LAST label word in the answer, or "" if there is none."""
    found = re.findall(r"\b(" + "|".join(labels) + r")\b", text.lower())
    return found[-1] if found else ""


def cost(tokens_in, tokens_out):
    return tokens_in / 1e6 * PRICE_IN + tokens_out / 1e6 * PRICE_OUT


def measure(name, build_prompt, system, ids=TEST):
    """Runs one prompt pattern on a list of tickets and prints the error rate with its four facts."""
    rows, tin, tout = [], 0, 0
    for tid in ids:
        t = tickets[tid]
        r = ask(build_prompt(t), system)
        tin += r.usage.input_tokens
        tout += r.usage.output_tokens
        rows.append({"ticket": tid, "gold": t["gold_label"], "second": t["ambiguous_with"],
                     "got": parse(answer_text(r)), "answer": answer_text(r)})
    strict = sum(x["got"] != x["gold"] for x in rows)
    either = sum(x["got"] not in (x["gold"], x["second"]) for x in rows)
    print(f"{name}: strict {strict}/{len(rows)}, either {either}/{len(rows)}, "
          f"{tin} in + {tout} out tokens, ${cost(tin, tout):.4f} | {MODEL}, {datetime.date.today()}")
    return {"name": name, "strict": strict, "either": either, "in": tin, "out": tout, "rows": rows}

## 1. Your prompt from Exercise 1

In [ ]:
# Paste SYSTEM_CONF from Exercise 1, step 7: the eight labels, and the two-line
# answer format (Label, then Confidence).

# Paste parse_confidence from Exercise 1, step 7.



In [ ]:
# GIVEN: your prompt on the sixty new tickets.
NEW = [f"T{i:03d}" for i in range(61, 121)]
rows = []
for tid in NEW:
    t = tickets[tid]
    text = answer_text(ask(text_of(t), SYSTEM_CONF))
    got = parse(text)
    rows.append({"ticket_id": tid, "subject": t["subject"], "gold_label": t["gold_label"],
                 "ambiguous_with": t["ambiguous_with"], "predicted": got, "confidence": parse_confidence(text),
                 "correct": got in (t["gold_label"], t["ambiguous_with"])})
print(len(rows), "tickets | errors (either label):", sum(not r["correct"] for r in rows),
      f"| {MODEL}, {datetime.date.today()}")
print("predicted:", dict(Counter(r["predicted"] for r in rows)))

## 2. Is the stated confidence worth anything?

In [ ]:
# GIVEN: answers grouped by stated confidence. If a stated 0.9 means anything,
# about nine in ten of those answers should be right.
buckets = [(0.0, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]
print(f"{'stated confidence':18} {'answers':>8} {'right':>7}")
for lo, hi in buckets:
    group = [r for r in rows if r["confidence"] is not None and lo <= r["confidence"] < hi]
    right = sum(r["correct"] for r in group)
    print(f"{lo:.1f} to {min(hi, 1.0):.1f}        {len(group):>8} {f'{right}/{len(group)}' if group else '-':>7}")
print("no confidence given:", sum(r["confidence"] is None for r in rows))

In [ ]:
# YOUR ANSWER (two sentences, as comments): is the stated confidence calibrated on these
# tickets, and what, if anything, could you use it for?

## 3. Read what the labels could not hold

Read the subjects, then open a few full tickets with `print(text_of(tickets["T0xx"]))`.

In [ ]:
# GIVEN: every ticket predicted "other", and every ticket under 0.8 confidence.
for r in sorted(rows, key=lambda r: (r["predicted"] != "other", r["confidence"] or 0)):
    if r["predicted"] == "other" or (r["confidence"] or 0) < 0.8:
        print(f"{r['ticket_id']}  {r['predicted']:16} {r['confidence']}  {r['subject'][:70]}")

## 4. Propose the missing label

In [ ]:
# Name the kind of message the eight labels have no place for (one word or two, snake_case).

# Define it in ONE line that a colleague could apply to a new ticket without asking you.

# List the ticket ids you would move to it. Read them; do not guess from the subjects alone.



## 5. Re-measure with nine labels

In [ ]:
# GIVEN: your label added to the prompt and to the label list; the moved tickets scored against it.
LABELS9 = LABELS + [NEW_LABEL]
SYSTEM9 = SYSTEM_CONF.replace("other: anything else", f"{NEW_LABEL}: {DEFINITION}\nother: anything else")
right9 = 0
for r in rows:
    t = tickets[r["ticket_id"]]
    gold = NEW_LABEL if r["ticket_id"] in MOVED else t["gold_label"]
    got = parse(answer_text(ask(text_of(t), SYSTEM9)), LABELS9)
    right9 += got in (gold, t["ambiguous_with"])
print(f"with {NEW_LABEL}: {right9}/{len(rows)} right (either label), against "
      f"{sum(r['correct'] for r in rows)}/{len(rows)} with eight labels and your labels")
print(f"this notebook: {spent['in']} in + {spent['out']} out tokens, ${cost(spent['in'], spent['out']):.4f}")